In [1]:
##############################################
##############################################
### TO GET ACCESS TO FILE IN GOOGLE COLAB ###
##############################################
##############################################

# To get access to the data files from github in Google Colab
!git clone https://github.com/math869c/graph_representation_st457.git

# set the folder to get access to the data
import os
os.chdir('/content') # to avoid error if rerun
os.chdir('./graph_representation_st457') # gives them access to everything in the folder

Cloning into 'graph_representation_st457'...
remote: Enumerating objects: 117, done.
remote: Counting objects: 100% (117/117), done.
remote: Compressing objects: 100% (84/84), done.
remote: Total 117 (delta 52), reused 83 (delta 29), pack-reused 0 (from 0)
Receiving objects: 100% (117/117), 32.41 MiB | 8.17 MiB/s, done.
Resolving deltas: 100% (52/52), done.


In [3]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 2.9 MB/s eta 0:00:00


In [4]:
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
import torch.nn as nn
import torch
import torch.optim as optim
import json
import optuna
import time

# load from other folders
from helper_functions import *
from model_classes import *

In [5]:
# Stock prices
open_prices_interp = pd.read_csv('data_folder/open_prices_interp.csv', index_col=0)
# into numpy
x = open_prices_interp.to_numpy()
tickers_with_data = open_prices_interp.columns

# Load graph data
with open("data_folder/firm_industry.json", "r") as f:
    firm_industry_dict = json.load(f)
A = np.load("data_folder/adjacency_matrix.npy")
adj_matrix = torch.tensor(A, dtype=torch.float32)

# make training data
batch_size =32
X_train, y_train, X_val, y_val, X_test, y_test, sc, train_loader, val_loader, test_loader = create_data(x,
                                                                                                        batch_size =batch_size,
                                                                                                        flatten_data = False, # Should be True for LSTM, False for GTC and GAT
                                                                                                        flatten_time_features=False # Should be True for GAT, False for LSTM and GTC
                                                                                                        )

dict_adj_matrices = {'sector':    {'MSE':0, 'model':np.empty, 'matrix':adj_matrix[:,:,0].unsqueeze(-1), 'pred':np.empty},
                     'industry':  {'MSE':0, 'model':np.empty, 'matrix':adj_matrix[:,:,1].unsqueeze(-1), 'pred':np.empty},
                     'corre':     {'MSE':0, 'model':np.empty, 'matrix':adj_matrix[:,:,2].unsqueeze(-1), 'pred':np.empty},
                     'everything':{'MSE':0, 'model':np.empty, 'matrix':adj_matrix[:,:,:],               'pred':np.empty} }

torch.Size([984, 8, 460, 5])
torch.Size([984, 460])


# building optuna objective function for hyperparameter tuning

In [6]:
optuna.logging.set_verbosity(optuna.logging.WARNING)
def objective(trial, adj_key, metric_name):
    """
    trial:       Optuna gives this automatically — it's how you ask for hyperparameter suggestions
    adj_key:     which graph to use — 'sector', 'industry', 'corre', or 'everything'
    metric_name: which metric to optimise for — 'accuracy', 'f1', 'mcc', 'return_ratio', 'sharpe'
    """

    # Optuna suggests hyperparameter values for this trial
    # Each suggest_ call says: "pick me a value from this range"
    # Optuna gets smarter about which values to pick after each trial

    emb_dim  = trial.suggest_categorical('emb_dim', [16, 32, 64, 128])
    lr       = trial.suggest_float('lr', 1e-4, 1e-2, log=True)  # log=True = search on log scale, better for lr
    dropout  = trial.suggest_float('dropout', 0.1, 0.5)
    epochs   = trial.suggest_categorical('epochs', [10, 20, 30])

    # Build the TGC model with those hyperparameters
    A_loop = dict_adj_matrices[adj_key]['matrix']
    K_num_relations = A_loop.shape[-1]

    model = TGCModel(
        num_features  = F,
        emb_dim       = emb_dim,
        gcn_dim       = emb_dim,
        num_relations = K_num_relations
    )

    # Swap the fixed dropout in Header for the trial's dropout value
    # (overrides the hardcoded 0.2 in Nat's Header class)
    for layer in model.head.model:
        if isinstance(layer, nn.Dropout):
            layer.p = dropout

    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    # Train for the chosen number of epochs
    for epoch in range(epochs):
        train_one_epoch(model, train_loader, A_loop, optimizer, criterion)

    # Get predictions on VALIDATION set (not test — never peek at test during tuning)
    val_preds = predict(model, val_loader, A_loop).numpy()  # shape [T_val, N]

    # y_val is the true returns for the validation period
    metrics = compute_metrics(y_val, val_preds)

    # Return the metric Optuna should maximise
    return metrics[metric_name]

# Fine tune and save best model

In [8]:
N_TRIALS = 5  # change to 50 for real results

# Define F - number of features. This was missing and caused a TypeError in the LSTM layer.
# Based on X_train.shape which is (984, 8, 460, 5), the number of features is 5.
F = X_train.shape[-1] # X_train is available in kernel state

results = {}

for metric_name in ['MSE']: # I only care about one metric, which is MSE. Only optimize over 1 metric
    results[metric_name] = {}

    for adj_key in ['sector', 'industry', 'corre', 'everything']:
        print(f"\nOptimising {metric_name} on {adj_key} graph...")

        study = optuna.create_study(direction='maximize')
        study.optimize(lambda trial: objective(trial, adj_key, metric_name), n_trials=N_TRIALS)

        results[metric_name][adj_key] = {
            'best_value':  round(study.best_value, 4),
            'best_params': study.best_params
        }
        print(f"  Best {metric_name}: {study.best_value:.4f} | params: {study.best_params}")

final_models = {}
final_preds  = {}
saved_model_paths = {}

for metric_name in results:
    final_models[metric_name] = {}
    final_preds[metric_name]  = {}
    saved_model_paths[metric_name] = {}

    for adj_key in ['sector', 'industry', 'corre', 'everything']:
        print(f"\nRetraining best {metric_name} / {adj_key} model...")

        p      = results[metric_name][adj_key]['best_params']
        A_loop = dict_adj_matrices[adj_key]['matrix']
        K_num_relations = A_loop.shape[-1]

        model = TGCModel(
            num_features  = F,
            emb_dim       = p['emb_dim'],
            gcn_dim       = p['emb_dim'],
            num_relations = K_num_relations
        )

        for layer in model.head.model:
            if isinstance(layer, nn.Dropout):
                layer.p = p['dropout']

        criterion = nn.MSELoss()
        optimizer = torch.optim.Adam(model.parameters(), lr=p['lr'])

        best_val_mse = float("inf")
        best_state_dict = None

        for epoch in range(p['epochs']):
            train_loss = train_one_epoch(model, train_loader, A_loop, optimizer, criterion)
            val_loss   = evaluate(model, val_loader, A_loop, criterion)

            print(f"  Epoch {epoch+1}/{p['epochs']} | train loss {train_loss:.6f} | val loss {val_loss:.6f}")

            if val_loss < best_val_mse:
                best_val_mse = val_loss
                best_state_dict = copy.deepcopy(model.state_dict())

        # load best weights found during retraining
        model.load_state_dict(best_state_dict)

        # save model to disk
        model_path = os.path.join("Best_models", f"best_tgc_{adj_key}_mse.pt")
        torch.save({
            'model_state_dict': model.state_dict(),
            'best_params': p,
            'best_val_mse': best_val_mse,
            'adj_key': adj_key,
            'metric_name': metric_name,
            'num_features': F,
            'num_relations': K_num_relations
        }, model_path)

        preds = predict(model, test_loader, A_loop).numpy()

        final_models[metric_name][adj_key] = model
        final_preds[metric_name][adj_key]  = preds
        saved_model_paths[metric_name][adj_key] = model_path

        print(f"  Done. Best val MSE: {best_val_mse:.6f}")
        print(f"  Saved to: {model_path}")
        print(f"  Predictions shape: {preds.shape}")


Optimising MSE on sector graph...
  Best MSE: 2.8178 | params: {'emb_dim': 16, 'lr': 0.004300923343445644, 'dropout': 0.4768339326807226, 'epochs': 20}

Optimising MSE on industry graph...


[W 2026-04-19 15:14:18,190] Trial 4 failed with parameters: {'emb_dim': 64, 'lr': 0.0007711491705044879, 'dropout': 0.36030945044209794, 'epochs': 30} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/optuna/study/_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "/tmp/ipykernel_16383/2324920184.py", line 16, in <lambda>
    study.optimize(lambda trial: objective(trial, adj_key, metric_name), n_trials=N_TRIALS)
                                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_16383/2115291027.py", line 40, in objective
    train_one_epoch(model, train_loader, A_loop, optimizer, criterion)
  File "/content/graph_representation_st457/helper_functions.py", line 236, in train_one_epoch
    loss.backward()
  File "/usr/local/lib/python3.12/dist-packages/torch/_tensor.py", line 630, in backward
    torch.autograd.ba

KeyboardInterrupt: 

In [ ]:
from google.colab import files

for adj_key in ['sector', 'industry', 'corre', 'everything']:
    files.download(saved_model_paths['MSE'][adj_key])

# plot best model

In [ ]:
# EVALUATE ON TEST SET
test_scores = {}
for metric_name in final_preds:
    test_scores[metric_name] = {}
    for adj_key in final_preds[metric_name]:
        preds = final_preds[metric_name][adj_key]
        test_scores[metric_name][adj_key] = compute_metrics(y_test, preds)

# RESULTS TABLE
METRICS     = ['accuracy', 'f1', 'mcc', 'return_ratio', 'sharpe']
GRAPH_TYPES = ['sector', 'industry', 'corre', 'everything']
COLORS      = ['steelblue', 'darkorange', 'green', 'crimson']

print("\n" + "="*70)
print("FINAL TEST RESULTS — best Optuna config per (metric, graph)")
print("="*70)

rows = []
for metric_name in test_scores:
    for adj_key in test_scores[metric_name]:
        row = {'optimised_for': metric_name, 'graph': adj_key}
        row.update(test_scores[metric_name][adj_key])
        rows.append(row)

results_table = pd.DataFrame(rows).set_index(['optimised_for', 'graph'])
print(results_table.to_string())

print("\n--- Best test score per metric (across all graph types) ---")
for metric_name in test_scores:
    best_graph = max(test_scores[metric_name],
                     key=lambda g: test_scores[metric_name][g][metric_name])
    best_val   = test_scores[metric_name][best_graph][metric_name]
    print(f"  {metric_name:15s}: {best_val:.4f}  (graph = {best_graph})")


# PLOTS
fig, axes = plt.subplots(1, len(METRICS), figsize=(18, 5), sharey=False)
fig.suptitle("Test performance by metric and graph type\n(each model optimised for its own metric)", fontsize=13)

for ax, metric_name in zip(axes, METRICS):
    vals = [test_scores[metric_name][g][metric_name] for g in GRAPH_TYPES]
    bars = ax.bar(GRAPH_TYPES, vals, color=COLORS, alpha=0.85, edgecolor='black', linewidth=0.6)
    ax.set_title(metric_name, fontsize=11)
    ax.set_ylabel(metric_name)
    ax.set_xticks(range(len(GRAPH_TYPES)))
    ax.set_xticklabels(GRAPH_TYPES, rotation=30, ha='right', fontsize=9)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                f'{v:.3f}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.show()

heatmap_data = pd.DataFrame(index=GRAPH_TYPES, columns=METRICS, dtype=float)
for metric_name in METRICS:
    for adj_key in GRAPH_TYPES:
        heatmap_data.loc[adj_key, metric_name] = test_scores[metric_name][adj_key][metric_name]

fig, ax = plt.subplots(figsize=(10, 4))
im = ax.imshow(heatmap_data.values.astype(float), aspect='auto', cmap='RdYlGn')
ax.set_xticks(range(len(METRICS)));      ax.set_xticklabels(METRICS, fontsize=10)
ax.set_yticks(range(len(GRAPH_TYPES))); ax.set_yticklabels(GRAPH_TYPES, fontsize=10)
plt.colorbar(im, ax=ax)
ax.set_title("Test metric heatmap — each cell = best Optuna model for that column's metric")
for i in range(len(GRAPH_TYPES)):
    for j in range(len(METRICS)):
        ax.text(j, i, f"{heatmap_data.values[i, j]:.3f}",
                ha='center', va='center', fontsize=9, color='black')
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, len(METRICS), figsize=(20, 4))
fig.suptitle("Distribution of per-firm scores across graph types", fontsize=13)

for ax, metric_name in zip(axes, METRICS):
    ax.set_title(metric_name)
    for adj_key, color in zip(GRAPH_TYPES, COLORS):
        preds = final_preds[metric_name][adj_key]
        y_dir = (y_test > 0).astype(int)
        p_dir = (preds  > 0).astype(int)
        if metric_name == 'accuracy':
            per_firm = [(y_dir[:, i] == p_dir[:, i]).mean() for i in range(y_test.shape[1])]
        elif metric_name == 'f1':
            from sklearn.metrics import f1_score
            per_firm = [f1_score(y_dir[:, i], p_dir[:, i], average='macro', zero_division=0)
                        for i in range(y_test.shape[1])]
        elif metric_name == 'mcc':
            from sklearn.metrics import matthews_corrcoef
            per_firm = [matthews_corrcoef(y_dir[:, i], p_dir[:, i])
                        for i in range(y_test.shape[1])]
        elif metric_name == 'return_ratio':
            trade    = np.where(p_dir == 1, 1, -1)
            per_firm = (y_test * trade).sum(axis=0).tolist()
        elif metric_name == 'sharpe':
            trade    = np.where(p_dir == 1, 1, -1)
            rets     = y_test * trade
            mu       = rets.mean(axis=0)
            sigma    = rets.std(axis=0) + 1e-8
            per_firm = (mu / sigma * np.sqrt(252)).tolist()

        ax.hist(per_firm, bins=40, alpha=0.5, label=adj_key, color=color)

    ax.legend(fontsize=7)
    ax.set_xlabel(metric_name)
    ax.set_ylabel("# firms")

plt.tight_layout()
plt.show()

# RESULTS TABLE
METRICS     = ['accuracy', 'f1', 'mcc', 'return_ratio', 'sharpe']
GRAPH_TYPES = ['sector', 'industry', 'corre', 'everything']
COLORS      = ['steelblue', 'darkorange', 'green', 'crimson']

print("\n" + "="*70)
print("FINAL TEST RESULTS — best Optuna config per (metric, graph)")
print("="*70)

rows = []
for metric_name in test_scores:
    for adj_key in test_scores[metric_name]:
        row = {'optimised_for': metric_name, 'graph': adj_key}
        row.update(test_scores[metric_name][adj_key])
        rows.append(row)

results_table = pd.DataFrame(rows).set_index(['optimised_for', 'graph'])
print(results_table.to_string())

print("\n--- Best test score per metric (across all graph types) ---")
for metric_name in test_scores:
    best_graph = max(test_scores[metric_name],
                     key=lambda g: test_scores[metric_name][g][metric_name])
    best_val   = test_scores[metric_name][best_graph][metric_name]
    print(f"  {metric_name:15s}: {best_val:.4f}  (graph = {best_graph})")

# PLOTS
fig, axes = plt.subplots(1, len(METRICS), figsize=(18, 5), sharey=False)
fig.suptitle("Test performance by metric and graph type\n(each model optimised for its own metric)", fontsize=13)

for ax, metric_name in zip(axes, METRICS):
    vals = [test_scores[metric_name][g][metric_name] for g in GRAPH_TYPES]
    bars = ax.bar(GRAPH_TYPES, vals, color=COLORS, alpha=0.85, edgecolor='black', linewidth=0.6)
    ax.set_title(metric_name, fontsize=11)
    ax.set_ylabel(metric_name)
    ax.set_xticks(range(len(GRAPH_TYPES)))
    ax.set_xticklabels(GRAPH_TYPES, rotation=30, ha='right', fontsize=9)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                f'{v:.3f}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.show()

heatmap_data = pd.DataFrame(index=GRAPH_TYPES, columns=METRICS, dtype=float)
for metric_name in METRICS:
    for adj_key in GRAPH_TYPES:
        heatmap_data.loc[adj_key, metric_name] = test_scores[metric_name][adj_key][metric_name]

fig, ax = plt.subplots(figsize=(10, 4))
im = ax.imshow(heatmap_data.values.astype(float), aspect='auto', cmap='RdYlGn')
ax.set_xticks(range(len(METRICS)));      ax.set_xticklabels(METRICS, fontsize=10)
ax.set_yticks(range(len(GRAPH_TYPES))); ax.set_yticklabels(GRAPH_TYPES, fontsize=10)
plt.colorbar(im, ax=ax)
ax.set_title("Test metric heatmap — each cell = best Optuna model for that column's metric")
for i in range(len(GRAPH_TYPES)):
    for j in range(len(METRICS)):
        ax.text(j, i, f"{heatmap_data.values[i, j]:.3f}",
                ha='center', va='center', fontsize=9, color='black')
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, len(METRICS), figsize=(20, 4))
fig.suptitle("Distribution of per-firm scores across graph types", fontsize=13)

for ax, metric_name in zip(axes, METRICS):
    ax.set_title(metric_name)
    for adj_key, color in zip(GRAPH_TYPES, COLORS):
        preds = final_preds[metric_name][adj_key]
        y_dir = (y_test > 0).astype(int)
        p_dir = (preds  > 0).astype(int)
        if metric_name == 'accuracy':
            per_firm = [(y_dir[:, i] == p_dir[:, i]).mean() for i in range(y_test.shape[1])]
        elif metric_name == 'f1':
            from sklearn.metrics import f1_score
            per_firm = [f1_score(y_dir[:, i], p_dir[:, i], average='macro', zero_division=0)
                        for i in range(y_test.shape[1])]
        elif metric_name == 'mcc':
            from sklearn.metrics import matthews_corrcoef
            per_firm = [matthews_corrcoef(y_dir[:, i], p_dir[:, i])
                        for i in range(y_test.shape[1])]
        elif metric_name == 'return_ratio':
            trade    = np.where(p_dir == 1, 1, -1)
            per_firm = (y_test * trade).sum(axis=0).tolist()
        elif metric_name == 'sharpe':
            trade    = np.where(p_dir == 1, 1, -1)
            rets     = y_test * trade
            mu       = rets.mean(axis=0)
            sigma    = rets.std(axis=0) + 1e-8
            per_firm = (mu / sigma * np.sqrt(252)).tolist()

        ax.hist(per_firm, bins=40, alpha=0.5, label=adj_key, color=color)

    ax.legend(fontsize=7)
    ax.set_xlabel(metric_name)
    ax.set_ylabel("# firms")

plt.tight_layout()
plt.show()